# KnobNet — MERT Layer 실험
MERT-v1-95M의 어느 레이어 출력이 노브 추정에 최적인지 비교합니다.

**실험 조건**
- 레이어: 1 ~ 12
- Phase 1: 20 epoch (MERT frozen)
- Phase 2: 3 epoch (MERT unfreeze)
- **중단 후 재실행 시 자동으로 이어서 진행**

## 1. 환경 설치

In [ ]:
!pip install -q transformers soundfile torchaudio

## 2. GitHub 클론

In [ ]:
import os

GITHUB_REPO = "https://github.com/YOUR_USERNAME/KnobNet.git"  # <-- 수정
PROJECT_DIR = "/content/KnobNet"

if not os.path.exists(PROJECT_DIR):
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("현재 디렉터리:", os.getcwd())

## 3. Google Drive 마운트 & 데이터 압축 해제

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

# ── 압축 파일 경로 수정 (여러 개면 추가) ──────────────────────────────────────
ZIP_FILES = [
    "/content/drive/MyDrive/KnobNet/data.zip.001",
    # "/content/drive/MyDrive/KnobNet/data.zip.002",
]

DATA_DIR = Path(PROJECT_DIR) / "data"
DATA_DIR.mkdir(exist_ok=True)
!apt-get install -q p7zip-full

for zip_file in ZIP_FILES:
    print(f"압축 해제 중: {Path(zip_file).name} ...")
    !7z x "{zip_file}" -o"{DATA_DIR}" -y
    print("완료")

In [ ]:
import csv
from pathlib import Path

# ── 데이터 구조 & CSV 경로 진단 ───────────────────────────────────────────────
print("=== data/ 하위 디렉터리 ===")
!find {DATA_DIR} -type d | sort

print("\n=== samples.csv 위치 ===")
!find {DATA_DIR} -name "samples.csv"

# CSV 첫 3행 확인
csv_candidates = list(DATA_DIR.rglob("samples.csv"))
if csv_candidates:
    with open(csv_candidates[0], newline="") as f:
        rows = list(csv.DictReader(f))
    print(f"\n=== {csv_candidates[0]} (첫 3행) ===")
    for row in rows[:3]:
        print(dict(row))

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from dataset.loader import make_loaders

# ── wet 디렉터리 수정 ─────────────────────────────────────────────────────────
WET_DIR = "data/wet/black"  # <-- 수정

train_loader, val_loader = make_loaders(
    dataset_root = PROJECT_DIR,
    wet_dir      = WET_DIR,
    batch_size   = 16,
    val_split    = 0.2,
    num_workers  = 2,
)
print(f"train: {len(train_loader.dataset):,}  val: {len(val_loader.dataset):,}")

In [ ]:
LAYERS        = list(range(1, 13))   # <-- 수정 (예: [1, 4, 8, 12])
PHASE1_EPOCHS = 60                   # <-- 수정
PHASE2_EPOCHS = 3                    # <-- 수정

CKPT_ROOT = Path("/content/drive/MyDrive/KnobNet/layer_exp")  # <-- 수정
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"레이어: {LAYERS}")
print(f"Phase1={PHASE1_EPOCHS}ep  Phase2={PHASE2_EPOCHS}ep")
print(f"체크포인트: {CKPT_ROOT}")

## 5. 실험 설정

In [ ]:
import csv
import torch
import torch.nn as nn

from model.model import KnobNet
from utils.config import KNOB_PARAMS
from train.train import (
    run_epoch, evaluate_all,
    make_optimizer, load_checkpoint,
)
from train.cache import (
    cache_embeddings, make_loaders_cached,
    run_epoch_cached, evaluate_all_cached,
)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.L1Loss()
TOLERANCE = 0.1
print("device:", device)


def log_to_csv(csv_path, row: dict):
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def print_epoch(layer_idx, phase, epoch, total_ep, train_loss, metrics, improved):
    mark = " *" if improved else ""
    print(f"[layer={layer_idx:2d} | P{phase} ep{epoch:02d}/{total_ep}]  "
          f"train={train_loss:.4f}  val={metrics['val_loss']:.4f}{mark}")
    print("  MAE  " + "  ".join(f"{k}={v:.4f}" for k, v in metrics["mae"].items()))
    print("  Acc  " + "  ".join(f"{k}={v*100:.1f}%" for k, v in metrics["acc"].items()))


def is_layer_done(log_csv, phase2_epochs):
    if not log_csv.exists():
        return False
    with open(log_csv, newline="") as f:
        for row in csv.DictReader(f):
            if int(row["phase"]) == 2 and int(row["epoch"]) >= phase2_epochs:
                return True
    return False


def read_best_from_csv(log_csv, phase=2):
    with open(log_csv, newline="") as f:
        rows = [r for r in csv.DictReader(f) if int(r["phase"]) == phase]
    best = min(rows, key=lambda r: float(r["val_loss"]))
    return {
        "val_loss": float(best["val_loss"]),
        "mae": {k: float(best[f"mae_{k}"]) for k in KNOB_PARAMS},
        "acc": {**{k: float(best[f"acc_{k}"]) for k in KNOB_PARAMS},
                "all": float(best["acc_all"])},
    }


def run_phase(layer_idx, phase, epochs, model, log_csv, ckpt_latest,
              loader_train=None, loader_val=None, use_cache=False):
    optimizer = make_optimizer(model, lr=1e-3, phase=phase)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    _train_loader = loader_train if loader_train is not None else train_loader
    _val_loader   = loader_val   if loader_val   is not None else val_loader
    _run_epoch    = run_epoch_cached  if use_cache else run_epoch
    _eval_all     = evaluate_all_cached if use_cache else evaluate_all

    start_epoch = 1
    best_val    = float("inf")

    if ckpt_latest.exists():
        meta = torch.load(ckpt_latest, map_location="cpu")
        if meta["phase"] == phase:
            meta = load_checkpoint(ckpt_latest, model, optimizer, scheduler)
            start_epoch = meta["epoch"] + 1
            best_val    = meta.get("best_val", float("inf"))
            print(f"  [resume] layer={layer_idx} Phase{phase} epoch {meta['epoch']} → {start_epoch}부터 재개")
        else:
            load_checkpoint(ckpt_latest, model)

    if start_epoch > epochs:
        print(f"  [skip] layer={layer_idx} Phase{phase} 이미 완료")
        return

    for epoch in range(start_epoch, epochs + 1):
        train_loss = _run_epoch(model, _train_loader, optimizer, criterion, device, scaler)
        metrics    = _eval_all(model, _val_loader, criterion, device, tolerance=TOLERANCE)
        val_loss   = metrics["val_loss"]
        scheduler.step()

        improved = val_loss < best_val
        if improved:
            best_val = val_loss

        torch.save({
            "phase": phase, "epoch": epoch, "best_val": best_val,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "val_loss":        val_loss,
        }, ckpt_latest)

        print_epoch(layer_idx, phase, epoch, epochs, train_loss, metrics, improved)
        log_to_csv(log_csv, {
            "layer": layer_idx, "phase": phase, "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_loss":   round(val_loss, 6),
            **{f"mae_{k}": round(v, 6) for k, v in metrics["mae"].items()},
            **{f"acc_{k}": round(v, 6) for k, v in metrics["acc"].items()},
        })


print("헬퍼 함수 정의 완료")

In [ ]:
import gc
import shutil

results = {}

for layer_idx in LAYERS:
    print(f"\n{'='*60}")
    print(f"  Layer {layer_idx:2d} / 12")
    print(f"{'='*60}")

    ckpt_latest = CKPT_ROOT / f"layer{layer_idx:02d}_latest.pt"
    log_csv     = CKPT_ROOT / f"layer{layer_idx:02d}_log.csv"

    # ── 이미 완료된 경우: CSV에서 결과 읽기 ──────────────────────────────
    if is_layer_done(log_csv, PHASE2_EPOCHS):
        print(f"  [skip] layer={layer_idx} 모든 Phase 완료 → CSV에서 로드")
        results[layer_idx] = read_best_from_csv(log_csv, phase=2)
        continue

    # ── 모델 초기화 ───────────────────────────────────────────────────────
    model = KnobNet(num_knobs=len(KNOB_PARAMS), freeze_mert=True,
                    layer_idx=layer_idx).to(device)

    # ── Phase 1: MERT frozen + 캐시 사용 ─────────────────────────────────
    phase1_done = (
        ckpt_latest.exists() and
        torch.load(ckpt_latest, map_location="cpu")["phase"] == 1 and
        torch.load(ckpt_latest, map_location="cpu")["epoch"] >= PHASE1_EPOCHS
    )
    if not phase1_done:
        print(f"  Phase 1  (MERT frozen, {PHASE1_EPOCHS} epoch, cache 사용)")
        layer_cache_dir = Path(PROJECT_DIR) / "cache" / f"layer{layer_idx:02d}"
        cache_embeddings(PROJECT_DIR, wet_dir=WET_DIR, layer_idx=layer_idx, device=str(device))
        train_loader_c, val_loader_c = make_loaders_cached(
            PROJECT_DIR, wet_dir=WET_DIR, layer_idx=layer_idx,
            batch_size=256, num_workers=2,
        )
        run_phase(layer_idx, 1, PHASE1_EPOCHS, model, log_csv, ckpt_latest,
                  loader_train=train_loader_c, loader_val=val_loader_c, use_cache=True)
        # Phase 1 완료 → 캐시 로더 + 캐시 파일 삭제
        del train_loader_c, val_loader_c
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        if layer_cache_dir.exists():
            shutil.rmtree(layer_cache_dir)
            print(f"  [cleanup] cache/layer{layer_idx:02d}/ 삭제 완료")
    else:
        print(f"  [skip] Phase 1 완료")
        load_checkpoint(ckpt_latest, model)

    # ── Phase 2: MERT unfreeze (일반 오디오 로더) ─────────────────────────
    print(f"  Phase 2  (MERT unfreeze, {PHASE2_EPOCHS} epoch)")
    model.unfreeze_mert()
    run_phase(layer_idx, 2, PHASE2_EPOCHS, model, log_csv, ckpt_latest)

    # ── 완료: checkpoint 삭제, 결과는 CSV에서 ────────────────────────────
    if ckpt_latest.exists():
        ckpt_latest.unlink()
        print(f"  [cleanup] {ckpt_latest.name} 삭제 완료")

    results[layer_idx] = read_best_from_csv(log_csv, phase=2)
    print(f"  [Layer {layer_idx}] 최종 MAE  "
          + "  ".join(f"{k}={v:.4f}" for k, v in results[layer_idx]["mae"].items()))
    print(f"  [Layer {layer_idx}] 최종 Acc  "
          + "  ".join(f"{k}={v*100:.1f}%" for k, v in results[layer_idx]["acc"].items()))

print("\n\n모든 레이어 실험 완료")

## 8. 결과 비교 테이블 & 그래프

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for layer_idx, m in sorted(results.items()):
    rows.append({
        "layer":     layer_idx,
        "drive_MAE": round(m["mae"]["drive"], 4),
        "level_MAE": round(m["mae"]["level"], 4),
        "tone_MAE":  round(m["mae"]["tone"],  4),
        "avg_MAE":   round(sum(m["mae"].values()) / len(m["mae"]), 4),
        "drive_Acc": round(m["acc"]["drive"], 3),
        "level_Acc": round(m["acc"]["level"], 3),
        "tone_Acc":  round(m["acc"]["tone"],  3),
        "avg_Acc":   round((m["acc"]["drive"]+m["acc"]["level"]+m["acc"]["tone"])/3, 3),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# CSV 저장
csv_path = CKPT_ROOT / "layer_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nCSV 저장: {csv_path}")

# 그래프
layers = df["layer"].tolist()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for col, label in [("drive_MAE","drive"),("level_MAE","level"),("tone_MAE","tone"),("avg_MAE","avg")]:
    ax1.plot(layers, df[col], marker="o", linestyle="--" if "avg" in col else "-", label=label)
ax1.set_xlabel("MERT Layer"); ax1.set_ylabel("MAE")
ax1.set_title("MAE vs MERT Layer"); ax1.set_xticks(layers)
ax1.legend(); ax1.grid(True, alpha=0.3)

for col, label in [("drive_Acc","drive"),("level_Acc","level"),("tone_Acc","tone"),("avg_Acc","avg")]:
    ax2.plot(layers, df[col], marker="o", linestyle="--" if "avg" in col else "-", label=label)
ax2.set_xlabel("MERT Layer"); ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy vs MERT Layer"); ax2.set_xticks(layers)
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = CKPT_ROOT / "layer_results.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"그래프 저장: {fig_path}")